# Handcrafted Image Retrieval Pipeline
## Oxford & Paris Buildings Datasets

**Pipeline:**
1. **RootSIFT** — Hellinger kernel for better descriptor comparison (Arandjelović & Zisserman)
2. **VLAD** — Vector of Locally Aggregated Descriptors (Jégou et al.)
3. **PCA + Whitening** — De-correlation and dimensionality reduction (Jégou & Chum)
4. **RANSAC Spatial Re-ranking** — Geometric verification (Philbin et al.)
5. **Average Query Expansion (AQE)** — Query enrichment (Chum et al.)

## 0. Setup & Imports

In [ ]:
import os
import sys
import time
import pickle
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import cv2
from pathlib import Path
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize

import scipy.io

print(f"OpenCV version: {cv2.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Python version: {sys.version}")
print()

# Check GPU availability (for future deep learning extensions)
import subprocess
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                            capture_output=True, text=True)
    print("GPUs available:")
    print(result.stdout)
except:
    print("No GPU info available")

## 1. Configuration

In [ ]:
# ─────────────────────────────────────────────
#  CONFIGURATION — adjust paths as needed
# ─────────────────────────────────────────────

# Dataset selection: 'oxford' or 'paris'
DATASET = 'oxford'  # change to 'paris' for Paris Buildings

# Kaggle input paths
BASE_INPUT = Path('/kaggle/input')

# Auto-detect dataset path
def find_dataset_path(base, dataset_name):
    candidates = list(base.glob(f'**/{dataset_name}*'))  + list(base.glob('**/*.jpg'))[:1]
    for p in base.iterdir():
        if dataset_name in p.name.lower():
            return p
    # Fallback: return first directory found
    dirs = [p for p in base.iterdir() if p.is_dir()]
    return dirs[0] if dirs else base

DATASET_ROOT = find_dataset_path(BASE_INPUT, DATASET)
print(f"Dataset root: {DATASET_ROOT}")

# Try to find images and ground truth
def find_subdir(root, names):
    for name in names:
        p = root / name
        if p.exists():
            return p
    # recursive search
    for name in names:
        matches = list(root.rglob(name))
        if matches:
            return matches[0]
    return root

IMAGES_DIR  = find_subdir(DATASET_ROOT, ['images', 'jpg', 'oxbuild_images', 'paris'])
GT_DIR      = find_subdir(DATASET_ROOT, ['gt_files_170407', 'gt', 'groundtruth', 'query'])

print(f"Images dir : {IMAGES_DIR}")
print(f"GT dir     : {GT_DIR}")

# Output / cache directory
OUTPUT_DIR = Path('/kaggle/working/retrieval_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR  = OUTPUT_DIR / 'cache'
CACHE_DIR.mkdir(exist_ok=True)

# ─────────────────────────────────────────────
#  HYPERPARAMETERS
# ─────────────────────────────────────────────
K_VLAD          = 256    # number of visual words for VLAD
PCA_DIM         = 256    # output dimension after PCA+whitening
RERANK_TOP_N    = 100    # images to spatially re-rank with RANSAC
AQE_TOP_K       = 10     # top images used for query expansion
MIN_INLIERS     = 6      # minimum RANSAC inliers to confirm a match
MAX_IMAGE_SIZE  = 1024   # resize largest side to this for SIFT extraction
N_WORKERS       = 4      # parallel workers for feature extraction

SIFT_N_FEATURES = 2000   # max SIFT keypoints per image

print(f"\nHyperparameters:")
print(f"  K_VLAD={K_VLAD}, PCA_DIM={PCA_DIM}")
print(f"  RERANK_TOP_N={RERANK_TOP_N}, AQE_TOP_K={AQE_TOP_K}")

## 2. Ground Truth Parsing

In [ ]:
def parse_oxford_gt(gt_dir: Path):
    """
    Parse Oxford/Paris ground truth files.
    Each landmark has 5 query files: *_query.txt, *_good.txt, *_ok.txt, *_junk.txt
    Returns list of dicts with keys: name, query_img, query_roi, good, ok, junk
    """
    queries = []
    query_files = sorted(gt_dir.glob('*_query.txt'))

    if not query_files:
        # Try .mat format (some versions)
        mat_files = list(gt_dir.glob('*.mat'))
        if mat_files:
            print(f"Found .mat files, trying to parse...")
        print(f"Warning: no query files found in {gt_dir}")
        return queries

    for qf in query_files:
        name = qf.stem.replace('_query', '')
        with open(qf) as f:
            line = f.read().strip()

        # Format: "oxc1_IMAGENAME x1 y1 x2 y2"  (Oxford)
        # or:     "IMAGENAME x1 y1 x2 y2"        (Paris)
        parts = line.split()
        img_name = parts[0].replace('oxc1_', '').replace('paris_', '')
        roi = list(map(float, parts[1:5])) if len(parts) >= 5 else None

        def read_list(suffix):
            p = gt_dir / f'{name}_{suffix}.txt'
            if not p.exists():
                return set()
            with open(p) as f:
                return set(l.strip().replace('oxc1_', '').replace('paris_', '')
                           for l in f if l.strip())

        queries.append({
            'name'      : name,
            'query_img' : img_name,
            'query_roi' : roi,
            'good'      : read_list('good'),
            'ok'        : read_list('ok'),
            'junk'      : read_list('junk'),
        })

    print(f"Loaded {len(queries)} queries from {gt_dir}")
    return queries


gt_queries = parse_oxford_gt(GT_DIR)
if gt_queries:
    print(f"\nSample query: {gt_queries[0]['name']}")
    print(f"  Image : {gt_queries[0]['query_img']}")
    print(f"  ROI   : {gt_queries[0]['query_roi']}")
    print(f"  #Good : {len(gt_queries[0]['good'])}, #OK: {len(gt_queries[0]['ok'])}")

## 3. Image List & Index

In [ ]:
# Collect all images in the dataset
all_image_paths = sorted(IMAGES_DIR.glob('*.jpg')) + \
                  sorted(IMAGES_DIR.glob('*.JPG')) + \
                  sorted(IMAGES_DIR.glob('*.png'))

# Build index: stem -> path
img_name_to_path = {p.stem: p for p in all_image_paths}
all_img_stems    = [p.stem for p in all_image_paths]
N_IMAGES         = len(all_image_paths)

print(f"Total images found: {N_IMAGES}")
print(f"First 5 stems: {all_img_stems[:5]}")

## 4. RootSIFT Extractor
> **Source:** Arandjelović & Zisserman — *Three things everyone should know to improve object retrieval*
> 
> RootSIFT: L1-normalize SIFT, then take element-wise square root.  
> Comparing RootSIFT with Euclidean distance ≡ Hellinger kernel on original SIFT.

In [ ]:
def extract_rootsift(image_path: Path,
                     max_size: int = MAX_IMAGE_SIZE,
                     n_features: int = SIFT_N_FEATURES):
    """
    Extract RootSIFT descriptors from an image.

    Steps (Arandjelović & Zisserman, 2012):
      1. Detect keypoints & compute SIFT descriptors
      2. L1-normalize each descriptor
      3. Element-wise square root  → RootSIFT

    Returns: (keypoints, descriptors) or ([], None) on failure
    """
    try:
        img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            return [], None

        # Resize to speed up computation
        h, w = img.shape
        scale = min(max_size / max(h, w), 1.0)
        if scale < 1.0:
            img = cv2.resize(img, (int(w * scale), int(h * scale)),
                             interpolation=cv2.INTER_AREA)

        # SIFT detection + description
        sift = cv2.SIFT_create(nfeatures=n_features,
                               contrastThreshold=0.04,
                               edgeThreshold=10)
        kps, descs = sift.detectAndCompute(img, None)

        if descs is None or len(descs) == 0:
            return [], None

        descs = descs.astype(np.float32)

        # ── RootSIFT transformation ────────────────────────────────────
        # Step 1: L1 normalization  (eps avoids division-by-zero)
        l1_norms = descs.sum(axis=1, keepdims=True) + 1e-7
        descs /= l1_norms
        # Step 2: element-wise square root
        descs = np.sqrt(descs)
        # Result is already L2-normalized: ||sqrt(x)||_2 = 1  since sum(x)=1
        # ──────────────────────────────────────────────────────────────

        return kps, descs

    except Exception as e:
        print(f"Error processing {image_path.name}: {e}")
        return [], None


def extract_rootsift_in_roi(image_path: Path, roi, max_size=MAX_IMAGE_SIZE, n_features=SIFT_N_FEATURES):
    """
    Extract RootSIFT only within a bounding-box ROI (for query images).
    roi = [x1, y1, x2, y2]  in original image coordinates.
    """
    kps, descs = extract_rootsift(image_path, max_size, n_features)
    if descs is None or roi is None:
        return kps, descs

    # Scale ROI to resized image
    img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return kps, descs
    h, w = img.shape
    scale = min(max_size / max(h, w), 1.0)
    x1, y1, x2, y2 = [c * scale for c in roi]

    mask = np.array([
        (x1 <= kp.pt[0] <= x2) and (y1 <= kp.pt[1] <= y2)
        for kp in kps
    ], dtype=bool)

    kps_filtered   = [kp for kp, m in zip(kps, mask) if m]
    descs_filtered = descs[mask] if mask.any() else None
    return kps_filtered, descs_filtered


# Quick sanity check
if all_image_paths:
    kps, descs = extract_rootsift(all_image_paths[0])
    print(f"Sanity check — {all_image_paths[0].name}:")
    print(f"  Keypoints : {len(kps)}")
    print(f"  Descriptors shape: {descs.shape if descs is not None else 'None'}")

## 5. Build Visual Vocabulary (K-Means)
> Used as codebook for VLAD encoding.

In [ ]:
VOCAB_CACHE = CACHE_DIR / f'vocabulary_K{K_VLAD}.pkl'

def build_vocabulary(image_paths, k=K_VLAD, max_desc_per_image=200,
                     sample_images=None, cache_path=None):
    """
    Build a visual vocabulary using MiniBatchKMeans.
    sample_images: use a random subset of images (faster for large datasets).
    """
    if cache_path and cache_path.exists():
        print(f"Loading cached vocabulary from {cache_path}")
        with open(cache_path, 'rb') as f:
            return pickle.load(f)

    print(f"Building vocabulary with K={k}...")
    t0 = time.time()

    if sample_images is not None:
        rng = np.random.default_rng(42)
        idxs = rng.choice(len(image_paths), size=min(sample_images, len(image_paths)), replace=False)
        paths_to_use = [image_paths[i] for i in idxs]
    else:
        paths_to_use = image_paths

    all_descs = []
    for path in tqdm(paths_to_use, desc="Extracting descriptors for vocabulary"):
        _, descs = extract_rootsift(path)
        if descs is not None and len(descs) > 0:
            # Subsample to speed up
            if len(descs) > max_desc_per_image:
                idx = np.random.choice(len(descs), max_desc_per_image, replace=False)
                descs = descs[idx]
            all_descs.append(descs)

    if not all_descs:
        raise ValueError("No descriptors found — check image paths!")

    all_descs = np.vstack(all_descs).astype(np.float32)
    print(f"  Total descriptors for clustering: {len(all_descs):,}")

    kmeans = MiniBatchKMeans(n_clusters=k,
                             random_state=42,
                             batch_size=min(10_000, len(all_descs)),
                             max_iter=300,
                             n_init=5,
                             verbose=0)
    kmeans.fit(all_descs)

    print(f"  Done in {time.time()-t0:.1f}s")

    if cache_path:
        with open(cache_path, 'wb') as f:
            pickle.dump(kmeans, f)
        print(f"  Saved vocabulary to {cache_path}")

    return kmeans


# Build / load vocabulary — use at most 2000 images for speed
vocabulary = build_vocabulary(
    all_image_paths,
    k=K_VLAD,
    sample_images=min(2000, N_IMAGES),
    cache_path=VOCAB_CACHE
)
CENTROIDS = vocabulary.cluster_centers_.astype(np.float32)  # (K, 128)
print(f"Vocabulary shape: {CENTROIDS.shape}")

## 6. VLAD Encoding
> **Source:** Jégou et al. — *Aggregating local descriptors into a compact image representation* (CVPR 2010)
>
> For each visual word $c_i$, accumulate the residuals of assigned descriptors:
> $v_{i,j} = \sum_{x: NN(x)=c_i} (x_j - c_{i,j})$
> Then L2-normalize the full concatenated vector.

In [ ]:
def vlad_encode(descriptors: np.ndarray, centroids: np.ndarray,
                power_norm: bool = True) -> np.ndarray:
    """
    Encode a set of local descriptors into a VLAD vector.

    Args:
        descriptors : (N, D) array of RootSIFT descriptors
        centroids   : (K, D) visual vocabulary centers
        power_norm  : apply signed square-root (SSR/intra-normalization)
                      as in Jégou & Chum (ECCV 2012) to reduce burstiness

    Returns:
        vlad : (K*D,) L2-normalized VLAD vector
    """
    K, D = centroids.shape
    N    = len(descriptors)

    # Assign each descriptor to nearest centroid
    # Use FLANN for speed when K is large
    diff_matrix = descriptors[:, np.newaxis, :] - centroids[np.newaxis, :, :]  # (N, K, D)
    dists = np.sum(diff_matrix ** 2, axis=2)   # (N, K)
    assignments = np.argmin(dists, axis=1)      # (N,)

    # Accumulate residuals
    vlad = np.zeros((K, D), dtype=np.float32)
    for k in range(K):
        mask = assignments == k
        if mask.any():
            vlad[k] = np.sum(descriptors[mask] - centroids[k], axis=0)

    # ── Power normalization (signed square root) ───────────────────────
    # Reduces the effect of bursty visual elements (Jégou & Chum 2012)
    if power_norm:
        vlad = np.sign(vlad) * np.sqrt(np.abs(vlad))
    # ──────────────────────────────────────────────────────────────────

    vlad = vlad.flatten()                    # (K*D,)

    # L2-normalize
    norm = np.linalg.norm(vlad)
    if norm > 1e-8:
        vlad /= norm

    return vlad


def vlad_encode_fast(descriptors: np.ndarray, centroids: np.ndarray,
                     power_norm: bool = True) -> np.ndarray:
    """
    Faster VLAD encoding using vectorized numpy operations.
    Preferred for large descriptor sets.
    """
    K, D = centroids.shape

    # Assignment via matrix distance
    # ||x - c||^2 = ||x||^2 - 2x·c + ||c||^2
    desc_sq   = np.sum(descriptors ** 2, axis=1, keepdims=True)          # (N,1)
    cent_sq   = np.sum(centroids   ** 2, axis=1, keepdims=True).T        # (1,K)
    cross     = descriptors @ centroids.T                                  # (N,K)
    dists     = desc_sq + cent_sq - 2 * cross                             # (N,K)
    assignments = np.argmin(dists, axis=1)                                 # (N,)

    # Vectorized residual accumulation
    vlad = np.zeros((K, D), dtype=np.float32)
    np.add.at(vlad, assignments, descriptors - centroids[assignments])

    if power_norm:
        vlad = np.sign(vlad) * np.sqrt(np.abs(vlad))

    vlad = vlad.flatten()
    norm = np.linalg.norm(vlad)
    if norm > 1e-8:
        vlad /= norm
    return vlad


# Quick test
if all_image_paths:
    _, test_descs = extract_rootsift(all_image_paths[0])
    if test_descs is not None:
        test_vlad = vlad_encode_fast(test_descs, CENTROIDS)
        print(f"VLAD vector shape : {test_vlad.shape}  (K={K_VLAD}, D=128)")
        print(f"L2 norm           : {np.linalg.norm(test_vlad):.4f}  (should be ~1.0)")

## 7. Feature Extraction for All Database Images

In [ ]:
VLAD_CACHE = CACHE_DIR / f'vlad_raw_K{K_VLAD}.npy'
NAMES_CACHE = CACHE_DIR / f'image_names.pkl'

def extract_all_vlad(image_paths, centroids, cache_vlad=None, cache_names=None):
    """
    Extract VLAD descriptors for all images. Caches to disk if cache paths given.
    Returns: (vlad_matrix [N, K*D], image_stems [N])
    """
    if cache_vlad and cache_vlad.exists() and cache_names and cache_names.exists():
        print(f"Loading cached VLAD descriptors...")
        vlad_matrix = np.load(cache_vlad)
        with open(cache_names, 'rb') as f:
            stems = pickle.load(f)
        print(f"  Loaded {len(vlad_matrix)} VLAD vectors, dim={vlad_matrix.shape[1]}")
        return vlad_matrix, stems

    print(f"Extracting VLAD for {len(image_paths)} images...")
    t0 = time.time()

    vlad_dim = K_VLAD * 128
    vlad_matrix = np.zeros((len(image_paths), vlad_dim), dtype=np.float32)
    stems = []
    failed = 0

    def process_one(args):
        idx, path = args
        _, descs = extract_rootsift(path)
        if descs is not None and len(descs) > 0:
            v = vlad_encode_fast(descs, centroids)
        else:
            v = np.zeros(vlad_dim, dtype=np.float32)
        return idx, path.stem, v

    args = list(enumerate(image_paths))

    with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
        futures = {executor.submit(process_one, a): a for a in args}
        for future in tqdm(as_completed(futures), total=len(args),
                           desc="Extracting VLAD"):
            try:
                idx, stem, v = future.result()
                vlad_matrix[idx] = v
                stems.append((idx, stem))
            except Exception as e:
                failed += 1

    # Reorder stems by index
    stems.sort(key=lambda x: x[0])
    stems = [s for _, s in stems]

    print(f"  Done in {time.time()-t0:.1f}s  ({failed} failed)")

    if cache_vlad:
        np.save(cache_vlad, vlad_matrix)
        with open(cache_names, 'wb') as f:
            pickle.dump(stems, f)
        print(f"  Saved VLAD cache to {cache_vlad}")

    return vlad_matrix, stems


vlad_raw, db_stems = extract_all_vlad(
    all_image_paths, CENTROIDS,
    cache_vlad=VLAD_CACHE,
    cache_names=NAMES_CACHE
)
print(f"\nVLAD matrix shape: {vlad_raw.shape}")

## 8. PCA + Whitening
> **Source:** Jégou & Chum — *Negative evidences and co-occurrences in image retrieval: the benefit of PCA and whitening* (ECCV 2012)
>
> Whitening: $\hat{X} = \text{diag}(\lambda_1^{-1/2}, \ldots, \lambda_{D_0}^{-1/2}) P^\top X$  then L2-renormalize.
> This de-correlates co-occurring visual words and reduces burstiness.

In [ ]:
PCA_CACHE = CACHE_DIR / f'pca_whitening_dim{PCA_DIM}_K{K_VLAD}.pkl'

def fit_pca_whitening(vlad_matrix, n_components=PCA_DIM, cache_path=None):
    """
    Fit PCA + Whitening on VLAD vectors.

    The whitening corresponds to computing:
        X_white = diag(eigenvalues^{-0.5}) @ eigenvectors.T @ (X - mean)
    then L2-renormalizing each row.

    Returns: fitted PCA object with whitening baked in.
    """
    if cache_path and cache_path.exists():
        print(f"Loading cached PCA+whitening from {cache_path}")
        with open(cache_path, 'rb') as f:
            return pickle.load(f)

    print(f"Fitting PCA (n_components={n_components}) + whitening...")
    t0 = time.time()

    # sklearn PCA with whiten=True applies the 1/sqrt(eigenvalue) scaling
    pca = PCA(n_components=n_components, whiten=True, random_state=42)
    pca.fit(vlad_matrix)

    print(f"  Variance explained: {pca.explained_variance_ratio_.sum()*100:.1f}%")
    print(f"  Done in {time.time()-t0:.1f}s")

    if cache_path:
        with open(cache_path, 'wb') as f:
            pickle.dump(pca, f)
        print(f"  Saved PCA to {cache_path}")

    return pca


def apply_pca_whitening(vlad_matrix, pca):
    """
    Project VLAD vectors through PCA+whitening, then L2-renormalize.
    """
    projected = pca.transform(vlad_matrix)         # (N, PCA_DIM)
    projected = normalize(projected, norm='l2')     # critical re-normalization step
    return projected.astype(np.float32)


pca_model = fit_pca_whitening(vlad_raw, n_components=PCA_DIM, cache_path=PCA_CACHE)

# Apply to all database images
print("Projecting database VLAD vectors...")
db_vlad = apply_pca_whitening(vlad_raw, pca_model)   # (N, PCA_DIM)
print(f"Projected VLAD matrix shape: {db_vlad.shape}")

# Sanity check: norms should be ~1.0
norms = np.linalg.norm(db_vlad, axis=1)
print(f"Norm stats — mean={norms.mean():.4f}, std={norms.std():.4f}")

## 9. Retrieval Engine
> Cosine similarity via dot product (vectors are L2-normalized).

In [ ]:
# Map stem -> index for fast lookup
stem_to_idx = {stem: i for i, stem in enumerate(db_stems)}


def encode_query(image_path: Path, roi=None) -> np.ndarray:
    """
    Encode a query image (optionally with ROI) into a PCA+whitened VLAD vector.
    """
    if roi is not None:
        kps, descs = extract_rootsift_in_roi(image_path, roi)
    else:
        kps, descs = extract_rootsift(image_path)

    if descs is None or len(descs) == 0:
        return np.zeros(PCA_DIM, dtype=np.float32)

    vlad = vlad_encode_fast(descs, CENTROIDS)
    vlad = apply_pca_whitening(vlad[np.newaxis], pca_model)[0]
    return vlad


def retrieve(query_vec: np.ndarray, db_matrix: np.ndarray, top_n: int = None):
    """
    Retrieve images by cosine similarity (dot product of L2-normalized vectors).

    Returns: indices sorted by descending similarity
    """
    sims = db_matrix @ query_vec          # (N,) cosine similarities
    ranked = np.argsort(-sims)            # descending order
    if top_n:
        ranked = ranked[:top_n]
    return ranked, sims[ranked]


print("Retrieval engine ready.")

## 10. RANSAC Spatial Re-ranking
> **Source:** Philbin et al. — *Object retrieval with large vocabularies and fast spatial matching* (CVPR 2007)
>
> Re-rank top-N candidates by verifying geometric consistency using RANSAC homography.

In [ ]:
def ransac_rerank(query_path: Path, query_roi,
                  candidate_stems, candidate_scores,
                  top_n=RERANK_TOP_N, min_inliers=MIN_INLIERS):
    """
    Re-rank candidates using RANSAC homography verification.

    For each of the top-N candidates:
      1. Extract RootSIFT from both query and candidate
      2. Match descriptors with FLANN / BF matcher
      3. Estimate homography with RANSAC
      4. Score = number of inliers (or 0 if < min_inliers)

    Candidates with >= min_inliers inliers are boosted above unverified ones.

    Returns: reranked list of stems
    """
    top_candidates = candidate_stems[:top_n]
    rest_candidates = candidate_stems[top_n:]

    # Extract query keypoints and descriptors
    q_kps, q_descs = extract_rootsift_in_roi(query_path, query_roi)
    if q_descs is None or len(q_kps) < 4:
        return list(candidate_stems)

    q_pts = np.float32([kp.pt for kp in q_kps])

    # FLANN-based matcher
    FLANN_INDEX_KDTREE = 1
    index_params  = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
    search_params = dict(checks=50)
    flann = cv2.FlannBasedMatcher(index_params, search_params)

    inlier_counts = {}

    for stem in top_candidates:
        if stem not in img_name_to_path:
            inlier_counts[stem] = 0
            continue
        cand_path = img_name_to_path[stem]
        c_kps, c_descs = extract_rootsift(cand_path)

        if c_descs is None or len(c_kps) < 4:
            inlier_counts[stem] = 0
            continue

        c_pts = np.float32([kp.pt for kp in c_kps])

        try:
            # k-NN matching (k=2) for Lowe's ratio test
            matches = flann.knnMatch(q_descs, c_descs, k=2)

            # Lowe's ratio test (ratio = 0.75)
            good = []
            for m_n in matches:
                if len(m_n) == 2:
                    m, n = m_n
                    if m.distance < 0.75 * n.distance:
                        good.append(m)

            if len(good) < 4:
                inlier_counts[stem] = 0
                continue

            src_pts = np.float32([q_pts[m.queryIdx] for m in good]).reshape(-1, 1, 2)
            dst_pts = np.float32([c_pts[m.trainIdx] for m in good]).reshape(-1, 1, 2)

            # RANSAC homography estimation
            _, mask = cv2.findHomography(src_pts, dst_pts,
                                          cv2.RANSAC,
                                          ransacReprojThreshold=10.0)
            inliers = int(mask.sum()) if mask is not None else 0
            inlier_counts[stem] = inliers

        except Exception:
            inlier_counts[stem] = 0

    # Sort by inlier count (verified first), then by original score
    def sort_key(stem):
        cnt = inlier_counts.get(stem, 0)
        return (-int(cnt >= min_inliers), -cnt)

    reranked_top = sorted(top_candidates, key=sort_key)
    return reranked_top + list(rest_candidates)


print("RANSAC spatial re-ranker ready.")

## 11. Average Query Expansion (AQE)
> **Source:** Chum et al. — *Total Recall: Automatic Query Expansion* (ICCV 2007)
>
> Average the VLAD vectors of the top verified results to form a richer query, then re-retrieve.

In [ ]:
def average_query_expansion(query_vec: np.ndarray,
                             ranked_stems,
                             db_vlad_matrix: np.ndarray,
                             stem_to_idx: dict,
                             top_k: int = AQE_TOP_K) -> np.ndarray:
    """
    Average Query Expansion (AQE).

    Averages the query vector with the VLAD vectors of the top-k retrieved images
    (after RANSAC re-ranking), then L2-renormalizes the result.

    Source: Chum et al. (ICCV 2007), as simplified by Arandjelović & Zisserman (2012).

    Args:
        query_vec      : (D,) original query VLAD
        ranked_stems   : list of image stems in retrieval order
        db_vlad_matrix : (N, D) database VLAD matrix
        stem_to_idx    : mapping from stem to row index
        top_k          : number of top results to use for expansion

    Returns: (D,) expanded and renormalized query vector
    """
    expansion_vecs = [query_vec]

    for stem in ranked_stems[:top_k]:
        if stem in stem_to_idx:
            expansion_vecs.append(db_vlad_matrix[stem_to_idx[stem]])

    if len(expansion_vecs) == 1:
        return query_vec  # nothing to expand with

    expanded = np.mean(expansion_vecs, axis=0)
    norm = np.linalg.norm(expanded)
    if norm > 1e-8:
        expanded /= norm

    return expanded.astype(np.float32)


print("AQE module ready.")

## 12. Evaluation Metrics
> Precision@k, Average Precision, Mean Average Precision (mAP)

In [ ]:
def average_precision(ranked_stems, good_set, ok_set, junk_set):
    """
    Compute Average Precision following the Oxford Buildings protocol.

    - Good + OK  → positive (relevant)
    - Junk       → ignored (neither positive nor negative)
    - Absent     → negative

    Args:
        ranked_stems : ordered list of retrieved image stems
        good_set     : set of 'good' stems (positives)
        ok_set       : set of 'ok' stems (positives)
        junk_set     : set of 'junk' stems (ignored)

    Returns: AP score in [0, 1]
    """
    positive_set = good_set | ok_set
    N_pos = len(positive_set)
    if N_pos == 0:
        return 0.0

    ap = 0.0
    n_retrieved = 0   # count of non-junk images seen so far
    n_relevant  = 0   # count of relevant images seen so far

    for stem in ranked_stems:
        if stem in junk_set:
            continue                            # skip junk, don't count
        n_retrieved += 1
        if stem in positive_set:
            n_relevant  += 1
            precision_at_k = n_relevant / n_retrieved
            ap += precision_at_k

    ap /= N_pos
    return ap


def precision_at_k(ranked_stems, good_set, ok_set, junk_set, k=5):
    """
    Precision at rank k (ignoring junk images).
    """
    positive_set = good_set | ok_set
    n_relevant = 0
    n_seen = 0

    for stem in ranked_stems:
        if stem in junk_set:
            continue
        n_seen += 1
        if stem in positive_set:
            n_relevant += 1
        if n_seen == k:
            break

    return n_relevant / k if k > 0 else 0.0


print("Evaluation metrics ready.")

## 13. Full Retrieval Pipeline (Single Query)

In [ ]:
def run_query(query_info: dict,
              db_vlad_matrix: np.ndarray,
              use_ransac: bool = True,
              use_aqe: bool = True,
              verbose: bool = False):
    """
    Full retrieval pipeline for a single query:
      1. Encode query → PCA+whitened VLAD
      2. Retrieve by cosine similarity
      3. [Optional] RANSAC spatial re-ranking on top-N
      4. [Optional] AQE → re-retrieve

    Returns: (ranked_stems, ap_score, p@5_score)
    """
    q_img  = query_info['query_img']
    q_roi  = query_info['query_roi']
    good   = query_info['good']
    ok     = query_info['ok']
    junk   = query_info['junk']

    # Resolve image path
    if q_img in img_name_to_path:
        q_path = img_name_to_path[q_img]
    else:
        if verbose:
            print(f"  Query image not found: {q_img}")
        return [], 0.0, 0.0

    # Step 1: Encode query
    q_vec = encode_query(q_path, roi=q_roi)

    # Step 2: Initial retrieval
    ranked_idx, scores = retrieve(q_vec, db_vlad_matrix)
    ranked_stems = [db_stems[i] for i in ranked_idx]

    # Remove query image itself from ranking
    ranked_stems = [s for s in ranked_stems if s != q_img]

    # Step 3: RANSAC spatial re-ranking
    if use_ransac:
        ranked_stems = ransac_rerank(
            q_path, q_roi,
            ranked_stems, scores,
            top_n=RERANK_TOP_N
        )

    # Step 4: Average Query Expansion → re-retrieve
    if use_aqe:
        expanded_q = average_query_expansion(
            q_vec, ranked_stems, db_vlad_matrix, stem_to_idx, top_k=AQE_TOP_K
        )
        ranked_idx2, _ = retrieve(expanded_q, db_vlad_matrix)
        ranked_stems = [db_stems[i] for i in ranked_idx2 if db_stems[i] != q_img]

    # Compute metrics
    ap  = average_precision(ranked_stems, good, ok, junk)
    p5  = precision_at_k(ranked_stems, good, ok, junk, k=5)

    if verbose:
        print(f"  Query: {query_info['name']:30s}  AP={ap:.4f}  P@5={p5:.4f}")

    return ranked_stems, ap, p5


print("Full pipeline ready.")

## 14. Evaluate All Queries → mAP

In [ ]:
def evaluate_all(queries, db_vlad_matrix,
                 use_ransac=True, use_aqe=True):
    """
    Run the full pipeline on all queries and compute mAP.
    """
    if not queries:
        print("No queries loaded — skipping evaluation.")
        return 0.0, {}

    all_ap   = []
    all_p5   = []
    results  = {}

    print(f"Evaluating {len(queries)} queries  "
          f"[RANSAC={'on' if use_ransac else 'off'}, "
          f"AQE={'on' if use_aqe else 'off'}]")
    print("-" * 60)

    for q in tqdm(queries, desc="Queries"):
        ranked, ap, p5 = run_query(q, db_vlad_matrix,
                                    use_ransac=use_ransac,
                                    use_aqe=use_aqe,
                                    verbose=True)
        all_ap.append(ap)
        all_p5.append(p5)
        results[q['name']] = {'ranked': ranked, 'ap': ap, 'p5': p5}

    mAP   = float(np.mean(all_ap))
    mean_p5 = float(np.mean(all_p5))

    print("-" * 60)
    print(f"mAP  = {mAP:.4f}")
    print(f"P@5  = {mean_p5:.4f}")

    return mAP, results


# ─── BASELINE: no RANSAC, no AQE ───────────────────────────────────────────
print("=" * 60)
print("BASELINE (VLAD + PCA-Whitening only)")
print("=" * 60)
mAP_baseline, results_baseline = evaluate_all(
    gt_queries, db_vlad,
    use_ransac=False, use_aqe=False
)

In [ ]:
# ─── FULL PIPELINE: RANSAC + AQE ───────────────────────────────────────────
print("=" * 60)
print("FULL PIPELINE (VLAD + PCA-Whitening + RANSAC + AQE)")
print("=" * 60)
mAP_full, results_full = evaluate_all(
    gt_queries, db_vlad,
    use_ransac=True, use_aqe=True
)

print(f"\n{'='*50}")
print(f" Baseline mAP  : {mAP_baseline:.4f}")
print(f" Full pipeline mAP : {mAP_full:.4f}")
print(f" Improvement   : +{(mAP_full - mAP_baseline)*100:.2f}%")
print(f"{'='*50}")

## 15. Ablation Study

In [ ]:
ablation_results = {}

configs = [
    ('Baseline (VLAD+PCA)', False, False),
    ('+ RANSAC reranking', True,  False),
    ('+ AQE only',         False, True),
    ('+ RANSAC + AQE',     True,  True),
]

for label, ransac, aqe in configs:
    mAP_val, _ = evaluate_all(gt_queries, db_vlad,
                               use_ransac=ransac, use_aqe=aqe)
    ablation_results[label] = mAP_val
    print()

print("\n── Ablation Summary ──────────────────────")
for label, val in ablation_results.items():
    print(f"  {label:30s}: mAP = {val:.4f}")

## 16. Vocabulary Size Experiment

In [ ]:
vocab_sizes = [64, 128, 256, 512]
vocab_map   = {}

for k in vocab_sizes:
    print(f"\n── K={k} ──────────────────────────────")
    v_cache = CACHE_DIR / f'vocabulary_K{k}.pkl'
    d_cache = CACHE_DIR / f'vlad_raw_K{k}.npy'
    n_cache = CACHE_DIR / f'image_names.pkl'
    p_cache = CACHE_DIR / f'pca_whitening_dim{PCA_DIM}_K{k}.pkl'

    vocab_k    = build_vocabulary(all_image_paths, k=k,
                                   sample_images=min(1000, N_IMAGES),
                                   cache_path=v_cache)
    cents_k    = vocab_k.cluster_centers_.astype(np.float32)

    vlad_raw_k, _ = extract_all_vlad(all_image_paths, cents_k,
                                      cache_vlad=d_cache, cache_names=n_cache)

    pca_k      = fit_pca_whitening(vlad_raw_k, n_components=min(PCA_DIM, k*128),
                                    cache_path=p_cache)
    db_vlad_k  = apply_pca_whitening(vlad_raw_k, pca_k)

    mAP_k, _   = evaluate_all(gt_queries, db_vlad_k,
                               use_ransac=False, use_aqe=False)
    vocab_map[k] = mAP_k
    print(f"  K={k:4d}  →  mAP = {mAP_k:.4f}")

# Restore original
CENTROIDS = vocabulary.cluster_centers_.astype(np.float32)

## 17. Visualization

In [ ]:
def show_retrieval(query_info, ranked_stems, n_show=5, figsize=(20, 4)):
    """
    Visualize query image and top-N retrieved results.
    Green border = relevant, Red border = irrelevant.
    """
    positive_set = query_info['good'] | query_info['ok']
    junk_set     = query_info['junk']
    q_img        = query_info['query_img']
    roi          = query_info['query_roi']

    top_results = [s for s in ranked_stems if s not in junk_set][:n_show]

    fig, axes = plt.subplots(1, n_show + 1, figsize=figsize)
    fig.suptitle(f"Query: {query_info['name']}  |  mAP(this query) = "
                 f"{results_full.get(query_info['name'],{}).get('ap',0):.3f}",
                 fontsize=14, fontweight='bold')

    def load_img(stem):
        p = img_name_to_path.get(stem)
        if p is None:
            return np.zeros((100, 100, 3), dtype=np.uint8)
        img = cv2.imread(str(p))
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img is not None else np.zeros((100,100,3), dtype=np.uint8)

    # Query image
    ax = axes[0]
    q_rgb = load_img(q_img)
    ax.imshow(q_rgb)
    if roi:
        x1, y1, x2, y2 = roi
        h, w = q_rgb.shape[:2]
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                   linewidth=3, edgecolor='yellow',
                                   facecolor='none')
        ax.add_patch(rect)
    ax.set_title('QUERY', fontsize=11, fontweight='bold')
    ax.axis('off')

    # Top results
    for i, stem in enumerate(top_results):
        ax = axes[i + 1]
        img = load_img(stem)
        ax.imshow(img)
        is_rel = stem in positive_set
        color  = 'green' if is_rel else 'red'
        label  = f'#{i+1} ✓' if is_rel else f'#{i+1} ✗'
        ax.set_title(label, color=color, fontsize=11, fontweight='bold')
        for spine in ax.spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(4)
        ax.axis('off')

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"retrieval_{query_info['name']}.png",
                dpi=100, bbox_inches='tight')
    plt.show()


# Show results for a few queries
if gt_queries and results_full:
    # Sort by AP to show good and bad cases
    sorted_q = sorted(gt_queries,
                       key=lambda q: results_full.get(q['name'], {}).get('ap', 0),
                       reverse=True)

    print("── Best query ──")
    best_q = sorted_q[0]
    show_retrieval(best_q, results_full[best_q['name']]['ranked'])

    print("── Worst query ──")
    worst_q = sorted_q[-1]
    show_retrieval(worst_q, results_full[worst_q['name']]['ranked'])

    if len(sorted_q) > 2:
        print("── Median query ──")
        mid_q = sorted_q[len(sorted_q) // 2]
        show_retrieval(mid_q, results_full[mid_q['name']]['ranked'])

## 18. Results Dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Image Retrieval — Results Dashboard', fontsize=16, fontweight='bold')

# ── 1. Per-query AP bar chart ──────────────────────────────────────────────
ax = axes[0, 0]
if results_full:
    names = list(results_full.keys())
    aps   = [results_full[n]['ap'] for n in names]
    colors = ['steelblue' if ap >= np.median(aps) else 'tomato' for ap in aps]
    ax.barh(range(len(names)), aps, color=colors, edgecolor='white')
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=8)
    ax.axvline(np.mean(aps), color='black', linestyle='--', linewidth=1.5,
               label=f'mAP={np.mean(aps):.3f}')
    ax.legend()
    ax.set_xlabel('Average Precision')
    ax.set_title('Per-Query AP (Full Pipeline)')
    ax.set_xlim(0, 1)

# ── 2. Ablation bar chart ─────────────────────────────────────────────────
ax = axes[0, 1]
if ablation_results:
    labs = list(ablation_results.keys())
    vals = list(ablation_results.values())
    bars = ax.barh(labs, vals, color=['#4C72B0','#DD8452','#55A868','#C44E52'])
    for bar, val in zip(bars, vals):
        ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=10)
    ax.set_xlabel('mAP')
    ax.set_title('Ablation Study')
    ax.set_xlim(0, 1)

# ── 3. Vocabulary size vs mAP ─────────────────────────────────────────────
ax = axes[1, 0]
if vocab_map:
    ks   = list(vocab_map.keys())
    maps = list(vocab_map.values())
    ax.plot(ks, maps, 'o-', color='steelblue', linewidth=2, markersize=8)
    for k, m in zip(ks, maps):
        ax.annotate(f'{m:.3f}', (k, m), textcoords='offset points',
                    xytext=(0, 10), ha='center', fontsize=9)
    ax.set_xlabel('K (vocabulary size)')
    ax.set_ylabel('mAP')
    ax.set_title('Vocabulary Size vs mAP')
    ax.set_xscale('log', base=2)
    ax.grid(True, alpha=0.3)

# ── 4. AP distribution histogram ─────────────────────────────────────────
ax = axes[1, 1]
if results_full:
    all_ap_vals = [results_full[n]['ap'] for n in results_full]
    ax.hist(all_ap_vals, bins=15, color='steelblue', edgecolor='white', alpha=0.8)
    ax.axvline(np.mean(all_ap_vals), color='red', linestyle='--',
               label=f'Mean={np.mean(all_ap_vals):.3f}')
    ax.axvline(np.median(all_ap_vals), color='orange', linestyle=':',
               label=f'Median={np.median(all_ap_vals):.3f}')
    ax.set_xlabel('Average Precision')
    ax.set_ylabel('# Queries')
    ax.set_title('AP Distribution')
    ax.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'results_dashboard.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Dashboard saved to {OUTPUT_DIR / 'results_dashboard.png'}")

## 19. Final Summary

In [ ]:
print("╔══════════════════════════════════════════════════════════╗")
print("║          HANDCRAFTED IMAGE RETRIEVAL — SUMMARY           ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  Dataset        : {DATASET.upper():<40}║")
print(f"║  # Images       : {N_IMAGES:<40}║")
print(f"║  K_VLAD         : {K_VLAD:<40}║")
print(f"║  PCA dim        : {PCA_DIM:<40}║")
print(f"║  RANSAC top-N   : {RERANK_TOP_N:<40}║")
print(f"║  AQE top-k      : {AQE_TOP_K:<40}║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  Baseline mAP   : {mAP_baseline:.4f}{' '*35}║")
print(f"║  Full pipe mAP  : {mAP_full:.4f}{' '*35}║")
print(f"║  Gain           : +{(mAP_full-mAP_baseline)*100:.2f}%{' '*34}║")
print("╚══════════════════════════════════════════════════════════╝")

print("\nAblation:")
for label, val in ablation_results.items():
    print(f"  {label:35s}: {val:.4f}")

if vocab_map:
    best_k = max(vocab_map, key=vocab_map.get)
    print(f"\nBest vocabulary size: K={best_k}  (mAP={vocab_map[best_k]:.4f})")